# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VenkataVishnuVardhanReddy/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** 
One row in the dataset represents **one unique content item (page) belonging to a specific client over a trailing 90-day aggregation window**. 

**Time Window Details:**
- **Historical Features Window:** Represents search and user behavior metrics aggregated over the trailing 90-day period (e.g., `impressions_90d`, `clicks_90d`, `pageviews_90d`).
- **Outcome (Label) Window:** Represents the performance trend of the page, calculated by comparing the last 30 days of search impressions against the previous 30 days of search impressions (within or immediately following the historical window).

Let's verify that the grain (uniqueness of content_id) and size of the dataset match this definition:

In [1]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Verify grain: content_id must be unique (no duplicates)
is_grain_valid = df['content_id'].nunique() == len(df)
print(f"Is content_id unique (grain holds): {is_grain_valid}")
print(f"Total rows in dataset: {len(df)}")


Is content_id unique (grain holds): True
Total rows in dataset: 30000


## 2. Fields: feature / label / context / excluded

We classify every field in the dataset into one of the four categories to establish structural boundaries:

- **Features (Model Inputs):**
  - Search telemetry: `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `days_with_impressions`
  - Analytics/engagement: `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d`, `engagement_rate`, `scroll_rate`, `days_with_sessions`, `ai_traffic_pct`
  - Content age: `content_age_days`, `days_since_last_update`
  - Content metadata: `word_count`, `char_count`, `content_type`, `main_intent`
  - Search volume: `search_volume`, `competition`, `cpc`
- **Labels (Targets):**
  - `trend_direction` (and the derived `is_declining` flag)
- **Context (IDs & Grouping):**
  - `content_id`, `client_id`, `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier`
- **Excluded (Blocked):**
  - `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d` (derive outcome window data, constitute direct label leakage).
  - `provider_used`, `model_used` (represent internal production details, highly missing and introduce strong data bias).
  - `trend_pct` (contains direct outcome leakage info).

In [2]:
# Output the number of columns in each category to confirm full classification
features = [
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count', 'char_count',
    'search_volume', 'competition', 'cpc', 'content_type', 'main_intent'
]
labels = ['trend_direction']
contexts = [
    'content_id', 'client_id', 'age_tier', 'age_tier_order', 'freshness_tier',
    'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier'
]
excluded = [
    'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d',
    'sessions_last_30d', 'sessions_prev_30d', 'provider_used', 'model_used', 'trend_pct'
]

print(f"Total Columns Classified: {len(features) + len(labels) + len(contexts) + len(excluded)}")
print(f" - Features: {len(features)}")
print(f" - Labels:   {len(labels)}")
print(f" - Context:  {len(contexts)}")
print(f" - Excluded: {len(excluded)}")


Total Columns Classified: 43
 - Features: 24
 - Labels:   1
 - Context:  9
 - Excluded: 9


## 3. Verify it with queries (grain, counts, missing values, windows)

We execute quantitative validation to check missing value rates and date ranges, ensuring the dataset meets our structural expectations:

In [3]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Missingness Rates per Field:")
missing_rates = df.isnull().mean()
print(missing_rates[missing_rates > 0].to_string())

print("\nContent Age and Update Ranges:")
print(f"Content Age range (days):       {df['content_age_days'].min()} to {df['content_age_days'].max()}")
print(f"Days since last update range:  {df['days_since_last_update'].min()} to {df['days_since_last_update'].max()}")


Missingness Rates per Field:
search_volume        0.082267
competition          0.082267
competition_level    0.087000
cpc                  0.082267
main_intent          0.079133
word_count           0.256633
char_count           0.256633
provider_used        0.714600
model_used           0.191100
word_count_tier      0.256633
char_count_tier      0.256633
scroll_rate          0.004167
trend_pct            0.112933

Content Age and Update Ranges:
Content Age range (days):       90 to 564
Days since last update range:  1 to 373


## 4. Data limits

**Limitations of Search Performance Telemetry:**
1. **CTR Volatility at Low Impressions:** Click-through rate (CTR) is highly volatile and unreliable for low-impression pages. A single click on 2 impressions yields a 50% CTR, which represents noise, not signal.
2. **Missingness in Metadata:** Word counts and search volumes contain missing values (e.g., 25.6% missing word counts) which require robust imputation (median or zero) to avoid biased model training.
3. **Observational vs. Causal:** The data is purely observational. A high opportunity score suggests ranking decay and traffic exposure, but does not mathematically guarantee that a content refresh will recover the traffic (causality would require A/B testing).

Let's demonstrate the CTR volatility limit by checking standard deviations across impression tiers:

In [4]:
# Show standard deviation of CTR for low vs high impression pages
low_imp = df[df['impressions_90d'] < 10]
high_imp = df[df['impressions_90d'] >= 500]

print(f"CTR standard deviation for low-impression pages (< 10):   {low_imp['ctr'].std():.4f}")
print(f"CTR standard deviation for high-impression pages (>= 500): {high_imp['ctr'].std():.4f}")


CTR standard deviation for low-impression pages (< 10):   8.7954
CTR standard deviation for high-impression pages (>= 500): 0.3169


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.